## Machine Learning – Prédiction du prix des billets d’avion ##

### Tests Statistiques : ###

In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, chi2_contingency, ttest_ind, pearsonr


In [96]:

df = pd.read_csv('Clean_Dataset.csv')


#### Test 1 — Student (t-test) : Le prix diffère-t-il entre vols directs et vols avec escale ?

In [97]:
direct   = df[df["stops"] == "zero"]["price"]
avec_esc = df[df["stops"] != "zero"]["price"]


p_value = ttest_ind(direct, avec_esc).pvalue
print(f"p-value={p_value:.4e}")
print(f"Moyenne direct={direct.mean():.2f}€  |  avec escale={avec_esc.mean():.2f}€")
alpha = 0.05


if p_value < alpha:
    print("On rejette H0 : il existe une différence significative entre les deux classes")
else:
    print("On ne peut pas rejeter H0 : pas de différence significative")

p-value=0.0000e+00
Moyenne direct=9375.94€  |  avec escale=22459.00€
On rejette H0 : il existe une différence significative entre les deux classes


==> On conclut que les prix des vols directs et avec escale sont significativement différents.
Les vols directs et avec escales n’ont pas le même niveau de prix en moyenne (impact réel des escales sur le pricing).

#### Test 2 — ANOVA : Les compagnies aériennes influencent-elles le prix des billets ?


In [98]:

groups = [group["price"].values for name, group in df.groupby("airline")]
f_stat, p_value = f_oneway(*groups)
print("F-statistic:", f_stat)
print("p-value:", p_value)
alpha = 0.05

if p_value < alpha:
    print("On rejette H0 : il existe une différence significative entre les compagnies aériennes.")
else:
    print("On ne peut pas rejeter H0 : il n’y a pas de différence significative.")


F-statistic: 17194.402096092468
p-value: 0.0
On rejette H0 : il existe une différence significative entre les compagnies aériennes.


==> On conclut que la compagnie aérienne influence significativement le prix du billet.
Les compagnies appliquent des stratégies de pricing différentes (certaines sont plus chères que d’autres).

#### Test 3 — Student (t-test) : Le type de classe (Économy vs Business) influence-t-il le prix des billets ?

In [99]:
group1 = df[df["class"] == "Economy"]["price"]
group2 = df[df["class"] == "Business"]["price"]
p_value = ttest_ind(group1, group2).pvalue

alpha = 0.05

print("p-value:", p_value)

if p_value < alpha:
    print("On rejette H0 : il existe une différence significative entre les deux classes")
else:
    print("On ne peut pas rejeter H0 : pas de différence significative")

p-value: 0.0
On rejette H0 : il existe une différence significative entre les deux classes


==> On conclut que le type de classe a un impact significatif sur le prix.
La classe Business est significativement plus chère que Economy, ce qui confirme une stratégie de segmentation tarifaire claire.

#### Test 4 — Pearson : Le délai avant départ influence-t-il le prix des billets ?

In [100]:
p_value = pearsonr(df["days_left"], df["price"]).pvalue
print(f"p-value={p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis")
else:
    print("Fail to reject the null hypothesis")

p-value=0.0000e+00
Reject the null hypothesis


Le timing de réservation impacte le prix, ce qui confirme une logique de pricing dynamique (yield management) dans le transport aérien.

#### Test 5 — Chi2-contingency : Est-ce que le choix de la classe (Economy / Business) dépend du moment de réservation (days_left) ?

In [101]:
contingency = pd.crosstab(df['class'],df['days_left'])
p_value = chi2_contingency(contingency).pvalue
print(p_value)

alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis: variable class (Economy,Buisness) is dependent on the days left after reservation moment")
else:
    print("Fail to reject the null hypothesis: variables class and days_left are independent")

2.819153632455799e-76
Reject the null hypothesis: variable class (Economy,Buisness) is dependent on the days left after reservation moment


Il existe une dépendance entre la classe et le délai de réservation.
Les clients Business peuvent réserver plus tard (flexibilité, budget élevé)
Les clients Economy réservent plus tôt pour obtenir de meilleurs prix